# Đề bài về nhà

## Yêu cầu

- Tự viết code cho mô hình Linear Regression theo công thức đã được dạy trong buổi lý thuyết trên lớp.
- Tự viết hàm dự đoán.
- Huấn luyện cả mô hình của thư viện và mô hình mình tự viết.
- In ra các trọng số: w0, w1, w2, ..., wn của cả 2 mô hình đã huấn luyện để quan sát và so sánh.
- Dự đoán dữ liệu tập test bằng cả 2 mô hình (mô hình thư viện thì dùng hàm predict() của thư viện, mô hình tự viết dùng hàm dự đoán tự viết), in ra kết quả bằng Dataframe như trong bài thực hành trên lớp.
- Tính RMSE trên tập test cho cả 2 mô hình và so sánh.

## Dữ liệu

In [ ]:
#Cài đặt phiên bản scikit-learn cũ hơn để lấy dữ liệu về giá nhà ở Boston
# Docker image đã ghim sẵn scikit-learn==1.1.3 (xem requirements.txt) nên không cần cài lại.
# Trên Google Colab thì mới cần chạy:
# !pip install scikit-learn==1.1.3

Tập dữ liệu giá nhà ở Boston đã có sẵn trên sklearn, dữ liệu đã được chuẩn hóa và chia thành tập train, tập test

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math

from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score

# Đọc dữ liệu

Dữ liệu về giá nhà ở Boston được hỗ trợ bởi sklearn, đọc dữ liệu thông qua hàm `datasets.load_boston()`

Xem thêm các bộ dữ liệu khác tại https://scikit-learn.org/stable/datasets/index.html#toy-datasets.

Dữ liệu được chia thành các thành phần data và target như tập diabetes. Dữ liệu cũng đã được chuẩn hóa, chỉ cần gọi ra và huấn luyện

In [2]:
# lay du lieu dataset - du lieu ve giá nhà
dataset = datasets.load_boston()
print("Số chiều dữ liệu input: ", dataset.data.shape)
print("Số chiều dữ liệu target: ", dataset.target.shape)
print()

print("5 mẫu dữ liệu đầu tiên:")
print("input: ", dataset.data[:5])
print("target: ",dataset.target[:5])

Số chiều dữ liệu input:  (506, 13)
Số chiều dữ liệu target:  (506,)

5 mẫu dữ liệu đầu tiên:
input:  [[6.3200e-03 1.8000e+01 2.3100e+00 0.0000e+00 5.3800e-01 6.5750e+00
  6.5200e+01 4.0900e+00 1.0000e+00 2.9600e+02 1.5300e+01 3.9690e+02
  4.9800e+00]
 [2.7310e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 6.4210e+00
  7.8900e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9690e+02
  9.1400e+00]
 [2.7290e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 7.1850e+00
  6.1100e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9283e+02
  4.0300e+00]
 [3.2370e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 6.9980e+00
  4.5800e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9463e+02
  2.9400e+00]
 [6.9050e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 7.1470e+00
  5.4200e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9690e+02
  5.3300e+00]]
target:  [24.  21.6 34.7 33.4 36.2]


/usr/local/lib/python3.11/site-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function load_boston is deprecated; `load_boston` is deprecated in 1.0 and will be removed in 1.2.

    The Boston housing prices dataset has an ethical problem. You can refer to
    the documentation of this function for further details.

    The scikit-learn maintainers therefore strongly discourage the use of this
    dataset unless the purpose of the code is to study and educate about
    ethical issues in data science and machine learning.

    In this special case, you can fetch the dataset from the original
    source::

        import pandas as pd
        import numpy as np

        data_url = "http://lib.stat.cmu.edu/datasets/boston"
        raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
        data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
        target = raw_df.values[1::2, 2]

    Alternative datasets include the California housing dataset (i.e.

**Chia dữ liệu làm 2 phần training 362 mẫu và testing 80 mẫu**

In [3]:
# cat nho du lieu, lay 1 phan cho qua trinh thu nghiem,
# chia train test cac mau du lieu
# dataset_X = dataset.data[:, np.newaxis, 2]
dataset_X = dataset.data

dataset_X_train = dataset_X[:404]
dataset_y_train = dataset.target[:404]

dataset_X_test = dataset_X[405:]
dataset_y_test = dataset.target[405:]

# Xây dựng mô hình

## Xây dựng mô hình bằng thư viện

In [4]:
# Mô hình Linear Regression của thư viện sklearn
regr = linear_model.LinearRegression()

## Xây dựng mô hình Linear Regression tự viết

In [5]:
# Linear Regression tự viết theo công thức (normal equation):
#   thêm cột 1 để học hệ số tự do w0  ->  X_bar = [1, X]
#   w = (X_bar^T . X_bar)^(-1) . X_bar^T . y
def fit_linear_regression(X, y):
    X_bar = np.hstack((np.ones((X.shape[0], 1)), X))
    w = np.linalg.pinv(X_bar.T @ X_bar) @ X_bar.T @ y
    return w  # w[0] = w0 (bias), w[1:] = w1..wn

## Hàm test mô hình tự viết

In [6]:
# Hàm dự đoán tự viết: y_hat = w0 + w1*x1 + ... + wn*xn
def predict(X, w):
    X_bar = np.hstack((np.ones((X.shape[0], 1)), X))
    return X_bar @ w

# Huấn luyện mô hình

## Huấn luyện mô hình của thư viện

In [7]:
# Huấn luyện mô hình của thư viện và in trọng số
regr.fit(dataset_X_train, dataset_y_train)
print("Mô hình thư viện:")
print("w0 =", regr.intercept_)
print("[w1, ..., wn] =", regr.coef_)

Mô hình thư viện:
w0 = 30.077166922901938
[w1, ..., wn] = [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]


## Training mô hình bằng Linear regression tự viết

In [8]:
# Huấn luyện mô hình tự viết và in trọng số
w = fit_linear_regression(dataset_X_train, dataset_y_train)
print("Mô hình tự viết:")
print("w0 =", w[0])
print("[w1, ..., wn] =", w[1:])

Mô hình tự viết:
w0 = 30.077166920562895
[w1, ..., wn] = [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]


# Dự đoán các mẫu dữ liệu

## Dự đoán các mẫu dữ liệu theo mô hình của thư viện

In [9]:
# Dự đoán tập test bằng hàm predict() của thư viện
y_pred_lib = regr.predict(dataset_X_test)
y_pred_lib[:5]

array([ 3.78705697,  6.64054968, 21.31276497, 15.41271412, 23.65229756])

## Dự đoán các mẫu dữ liệu tính theo linear regression tự viết

In [10]:
# Dự đoán tập test bằng hàm tự viết, rồi so sánh: thực tế / thư viện / tự viết
y_pred_custom = predict(dataset_X_test, w)

pd.DataFrame(
    data=np.array([dataset_y_test, y_pred_lib, y_pred_custom]).T,
    columns=["Thực tế", "Dự đoán (thư viện)", "Dự đoán (tự viết)"],
)

,Thực tế,Dự đoán (thư viện),Dự đoán (tự viết)
0,5.0,3.787057,3.787057
1,11.9,6.640550,6.640550
2,27.9,21.312765,21.312765
3,17.2,15.412714,15.412714
4,27.5,23.652298,23.652298
...,...,...,...
96,22.4,23.755044,23.755044
97,20.6,22.081673,22.081673
98,23.9,28.181773,28.181773
99,22.0,26.572420,26.572420


## Đánh giá mô hình linear regression của thư viện

In [11]:
# RMSE của mô hình thư viện
rmse_lib = math.sqrt(mean_squared_error(dataset_y_test, y_pred_lib))
print("RMSE mô hình thư viện: {:.4f}".format(rmse_lib))

RMSE mô hình thư viện: 5.7495


## Đánh giá mô hình linear regression tự viết

In [12]:
# RMSE của mô hình tự viết (kỳ vọng xấp xỉ mô hình thư viện vì cùng là nghiệm OLS)
rmse_custom = math.sqrt(mean_squared_error(dataset_y_test, y_pred_custom))
print("RMSE mô hình tự viết:  {:.4f}".format(rmse_custom))
print("Chênh lệch RMSE giữa 2 mô hình: {:.6f}".format(abs(rmse_lib - rmse_custom)))

RMSE mô hình tự viết:  5.7495
Chênh lệch RMSE giữa 2 mô hình: 0.000000
